In [45]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim.lr_scheduler as lr_scheduler
from comet_ml import Experiment
from scipy import linalg
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from src.utils.samplers.data import DatasetSampler, PairedLoaderSampler
from src.utils.samplers.synthetic import StandardNormalSampler, SwissRollSampler
from src.utils.training.weather_notebook import (
    load_weather_tensors,
    paired_sampler_weather,
    unpaired_sampler_weather,
)

from src.utils.notebook_setup import ensure_repo_imports

REPO_ROOT = ensure_repo_imports()

tsne = TSNE(n_components=2, random_state=50)


EBiEOT-GMM: model and cost are built via Hydra (`compose_weather_cfg` / `build_gmm_model`) in the config cell below.


# Data preparation

### PS
1) Source  
$X \in \mathbb{R}^{N \times d_{1}}, N - \text{number of locations}, d_{1} - \text{features dim}$ \
$x = (\mu, \sigma) - \text{for a given location in June}$ \
$N = 1396, d_{1} = 188$ 

2) $Y \in \mathbb{R}^{N \times M \times d_{2}}, N - \text{number of locations}, M - \text{measurements for a given location in January by day}$ \
$M = [1, 31], d_{2} = 94$ 

In [3]:
##########################################
#-------------- RAW DATA -----------------
##########################################

import numpy as np
import pandas as pd

root = '../tabred/kal/weather'

data = np.load(f'{root}/X_num.npy')
data = np.stack([d for d in data if sum(np.isnan(d)) == 0])
data_csv = pd.read_csv(f'{root}/csv/X_num.csv')
#train_data = data[train_idx]
#test_data = data[test_idx]

target = np.load(f'{root}/Y.npy')
meta = np.load(f'{root}/X_meta.npy')
meta = np.stack([meta[i] for i, d in enumerate(data) if sum(np.isnan(d)) == 0])
meta_csv = pd.read_csv(f'{root}/csv/X_meta.csv')

names = list(data_csv.columns)
names.append('location')
data_new = np.concatenate((data, meta[:, -2].reshape(-1, 1)), axis=1)

In [4]:
#########################################################
#--------------- Month/location splitted ---------------- 
#########################################################
scaler = StandardScaler()

dict_location_src = {}
for d in data_new:
    if d[-2] == 1.0:
        d_new = d[:-7]
        try:
            dict_location_src[d[-1]].append(d_new)
        except KeyError:
            dict_location_src[d[-1]] = []
            dict_location_src[d[-1]].append(d_new)
     

dict_location_src_new = {}
for key in dict_location_src.keys():
    item = dict_location_src[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_src_new[key] = item
dict_location_src = dict_location_src_new
# ------------------------------------------------------------

dict_location_trg = {}
for d in data_new:
    if d[-2] == 6.0:
        d_new = d[:-7]
        try:
            dict_location_trg[d[-1]].append(d_new)
        except KeyError:
            dict_location_trg[d[-1]] = []
            dict_location_trg[d[-1]].append(d_new)
    
dict_location_trg_new = {}
for key in dict_location_trg.keys():
    item = dict_location_trg[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_trg_new[key] = item
dict_location_trg = dict_location_trg_new

print(len(dict_location_trg), len(dict_location_src))

1653 1578


In [5]:
#########################################################
#--------------------- X, Y paired ----------------------
#########################################################

chosen_locs = list(dict_location_trg.keys())[:200]
X_pair_orig, Y_pair_orig = [], []
for key in dict_location_src.keys():
    if key not in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_pair_orig.append(x)
    item_trg = dict_location_trg[key]
    Y_pair_orig.append(item_trg) # sample
X_pair_orig = np.stack(X_pair_orig)


#########################################################
#----------------------- X, Y ---------------------------
#########################################################

# N x 1 x 2D - src
# N x M x D - trg
 
# sampling: 
# b x 1 x 2D,
# b x M x D -> sample -> b x 1 x D

X_orig = []
for key in dict_location_src.keys():
    if key in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_orig.append(x)
X_orig = np.stack(X_orig)

Y_orig = []
for key in dict_location_trg.keys():
    if key in chosen_locs:
        continue
    item_trg = dict_location_trg[key]
    Y_orig.append(item_trg) # sample

In [6]:
print(X_orig.shape, len(Y_orig), X_pair_orig.shape, len(Y_pair_orig))

(1386, 188) 1453 (192, 188) 192


# Running

In [15]:
source_data = X_orig
target_data = Y_orig[0]
X_DIM = source_data.shape[1]
Y_DIM = target_data.shape[1]
#X_DIM = data_set["features"].shape[1]
#Y_DIM = data_set["features"].shape[1]
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 42

N_POTENTIALS = 10
M_POTENTIALS = 1 #10
EPSILON = 0.01
A_DIAGONAL_INIT = 0.5
L_PAIRED_SAMPLES = len(X_pair_orig)
M_X_UNPAIRED_SAMPLES = 0
N_Y_UNPAIRED_SAMPLES = 0

BATCH_SIZE = 32
SAMPLING_BATCH_SIZE = 128

D_LR = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float("inf")

NUM_LABELED = 10
TRAIN_SUBSET_SIZE = 2

PLOT_EVERY = 1000
MAX_STEPS = 20000
CONTINUE = -1

In [16]:
EXP_COST = "MLP_deep_deep"
EXP_COST_INCLUDED = True
EXP_META_INFO = ""
EXP_NAME = (
    f"EBiEOT-GMM_Batch_Effect_"
    + f"EPSILON_{EPSILON}_"
    + f"N_{N_POTENTIALS}_"
    + f"M_{M_POTENTIALS}_"
    + f"with_{EXP_COST}_"
    + f"cost_included_{EXP_COST_INCLUDED}_"
    + f"N_PAIRED_{NUM_LABELED}_"
    + f"M_UNPAIRED_{len(source_data)}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    N_PAIRED_SAMPLES=NUM_LABELED,
    M_UNPAIRED_SAMPLES=len(source_data),
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)


In [17]:
#pytorch_total_params = sum(p.numel() for p in D.parameters())
#pytorch_total_params

## Ablation Study

In [18]:
paired_sampler = paired_sampler_weather
unpaired_sampler = unpaired_sampler_weather


In [51]:
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)

from src.utils.training import CometExperiment, compose_weather_cfg, make_adam

EXPERIMENT = "gmm_weather"
OVERRIDES: list[str] = []
EXPERIMENT_ALIASES = {"gmm-weather": "gmm_weather"}

cfg, EXPERIMENT_KEY, seed = compose_weather_cfg(
    str(REPO_ROOT), str(EXPERIMENT), OVERRIDES, aliases=EXPERIMENT_ALIASES
)

device = torch.device(
    f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu"
)
torch.set_default_device(device)
dtype = torch.float64
torch.set_default_dtype(dtype)


In [52]:
    @torch.no_grad()
    def forward_test(model, batched_x: torch.Tensor, Y_pair_test) -> torch.Tensor:  # -> [bs]
        samples = []
        fids = []
        fids2 = []
        total_probs = []
        batch_size = 1
        sampling_batch_size = 1

        num_sampling_iterations = len(Y_pair_test)
        for i in range(num_sampling_iterations):
            sub_batch_x = batched_x[sampling_batch_size * i : sampling_batch_size * (i + 1)]
            sub_batch_y = torch.tensor(Y_pair_test[i]).to('cuda')
            log_w_n = model.log_w_n().to(dtype)
            a_n = model.a_n().to(dtype)
            A_n = model.A_n().to(dtype)
            
            cond_distr_paired = model.get_conditional_distribution(sub_batch_x.to(torch.float64),
                                                                   log_w_n.to(torch.float64),
                                                                   a_n.to(torch.float64),
                                                                   A_n.to(torch.float64))
            D_loss_paired = -cond_distr_paired.log_prob(sub_batch_y).mean()
            
            total_probs.append(D_loss_paired.cpu())
            
            samples = []
            for _ in range(len(sub_batch_y)):
                samples.append(cond_distr_paired.sample())
            fid_samples = np.array(torch.cat(samples, dim=0).cpu())
            fid_samples_2 = np.array(sub_batch_y.cpu())
            
            mu1 = np.mean(fid_samples, axis=0)
            sigma1 = np.cov(fid_samples, rowvar=False)
            mu2 = np.mean(fid_samples_2, axis=0)
            sigma2 = np.cov(fid_samples_2, rowvar=False)
            
            diff = mu1 - mu2
            covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
            tr_covmean = np.trace(covmean)
            fid = (diff.dot(diff) + np.trace(sigma1) +  np.trace(sigma2) - 2 * tr_covmean)
            fids.append(fid.real)
            fids2.append(fid.real / np.var(fid_samples_2))


        return samples, np.mean(total_probs), np.mean(fids), np.mean(fids2)

In [55]:
%load_ext autoreload
%autoreload 2

OUTPUT_SEED = 50
random.seed(OUTPUT_SEED)
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)
loader_kwargs = {"num_workers": 0, "pin_memory": True, "generator": torch.Generator(device='cpu')}

def mse(a, b):
    l = (a - b) ** 2
    return l.mean()

results_df = pd.DataFrame(columns=['L_PAIRED_SAMPLES', 'FOSCTTM_Score'])
MAX_STEPS = 10000
experiment = Experiment(project_name="inverse_ot")
experiment.set_name(EXP_NAME)
stats = []

# Splitting
L_PAIRED_SAMPLES = 90
L_UNPAIRED_SAMPLES = 500
X, Y = X_orig[:L_UNPAIRED_SAMPLES], Y_orig[-L_UNPAIRED_SAMPLES:]
X_pair, Y_pair = X_pair_orig[:L_PAIRED_SAMPLES], Y_pair_orig[:L_PAIRED_SAMPLES]
X_pair_test, Y_pair_test = X_pair_orig[-100:], Y_pair_orig[-100:]

print(len(X), len(Y), len(X_pair), len(Y_pair), len(X_pair_test), len(Y_pair_test))

for _ in [0]:
    test_size = 100
    print("Training with number of labeled:", L_PAIRED_SAMPLES)
    
    model = build_gmm_model(cfg, device).to(dtype)
    model.to('cuda')
    
    D_opt = torch.optim.Adam(model.parameters(), lr=4e-3)
    scheduler = lr_scheduler.StepLR(D_opt, step_size=1000, gamma=0.5) #0.87
     
    if CONTINUE > -1:
        D_opt.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_{CONTINUE}.pt")))
        
    for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):    
        # training loop
        D_opt.zero_grad()
    
        x_batch, y_batch = unpaired_sampler(X, Y, BATCH_SIZE)
        x_batch = x_batch.to(dtype)
        y_batch = y_batch.to(dtype)
        
        log_w_n = model.log_w_n().to(dtype)
        a_n = model.a_n().to(dtype)
        A_n = model.A_n().to(dtype)
        cond_distr_unpaired = model.get_conditional_distribution(
            x_batch.repeat(int(cfg.train.unpaired_batch_size), 1).to(torch.float64), 
            log_w_n.to(torch.float64), 
            a_n.to(torch.float64),
            A_n.to(torch.float64)
        )
        
        fwd = cond_distr_unpaired.log_prob(y_batch.repeat(int(cfg.train.unpaired_batch_size), 1))
        D_loss_unpaired = -torch.log(
            torch.mean(torch.exp(fwd.reshape(int(cfg.train.unpaired_batch_size), int(cfg.train.unpaired_batch_size))), dim=-1)
        ).mean()
    
        if EXP_COST_INCLUDED:
            x_pair_batch, y_pair_batch = paired_sampler(X_pair, Y_pair, BATCH_SIZE)
            cond_distr_paired = model.get_conditional_distribution(x_pair_batch.to(torch.float64),
                                                                   log_w_n.to(torch.float64),
                                                                   a_n.to(torch.float64),
                                                                   A_n.to(torch.float64))
            D_loss_paired = -cond_distr_paired.log_prob(y_pair_batch).mean()

            D_loss = D_loss_unpaired + D_loss_paired
            D_loss.backward()
            D_opt.step()
        
        
        if step % 1000 == 0:
            translated, total_probs, fids, fids2 = forward_test(model, 
                                           torch.tensor(X_pair_test).to('cuda').to(torch.float64),
                                            Y_pair_test,)
            print(-total_probs, fids, fids2)

    translated, total_probs, fids, fids2 = forward_test(model, 
                                           torch.tensor(X_pair_test).to('cuda').to(torch.float64),
                                            Y_pair_test,)
    foscttm_score = -total_probs
    
    new_row = pd.DataFrame({
        'L_PAIRED_SAMPLES': [L_PAIRED_SAMPLES],
        'FOSCTTM_Score': [foscttm_score],
        'fid': [fids],
        'vfid': [fids2]
    })
    results_df = pd.concat([results_df, new_row], ignore_index=True)
print(results_df)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
500 500 90 90 100 100
Training with number of labeled: 90


  0%|                                         | 8/10000 [00:02<40:16,  4.13it/s]

-458.10428349935637 119.11975650649933 1201.7808238754387


 10%|███▊                                  | 1009/10000 [00:21<13:09, 11.39it/s]

-0.22818229782202326 8.26295863658546 79.338925248303


...

 90%|██████████████████████████████████▏   | 9008/10000 [02:43<01:49,  9.07it/s]

32.60801949368183 7.140584061101573 68.38626798633726


100%|█████████████████████████████████████| 10000/10000 [02:58<00:00, 56.03it/s]


  L_PAIRED_SAMPLES  FOSCTTM_Score       fid       vfid
0               90      34.665192  7.118523  68.082497


/var/tmp/ipykernel_476081/354169610.py:100: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, new_row], ignore_index=True)


In [56]:
a = [0.34, 0.28, 0.34]
b = [7.14, 7.28, 7.11]
c = [68, 69, 68]
print(np.mean(a), np.std(a))
print(np.mean(b), np.std(b))
print(np.mean(c), np.std(c))

0.32000000000000006 0.0282842712474619
7.176666666666667 0.0740870359029763
68.33333333333333 0.4714045207910317


In [ ]:
#     |  Ours         | cGAn          | uGAn         |  CNF          |  Regres.
# FID | 7.21 +- 0.04  | 15.79 +- 1.11 | 15.44 +- 1.89| 18.72 +- 0.09 | 8.29 +- 0.044
# vFID| 72 +- 1       | 156 +- 11     | 152 +- 19    | 184 +- 1      | 81.0 +- 0.4